## Project: Building a Semantic Search Engine

# Mission

Build a semantic search engine for a topic of your choice. You’ll discover how chunking strategy affects search quality in your specific domain.

# What to Build

A working semantic search engine that demonstrates:

- *Domain expertise*: Choose content you understand so you can evaluate search quality
- *Chunking comparison*: Test different strategies and see which works best for your content type
- *Real semantic understanding*: Search by concept, theme, or meaning rather than exact keywords
- *Practical insights*: Discover what makes chunking effective in your specific domain

# Setup

Prerequisites
- Qdrant Cloud cluster (URL + API key)
- Python 3.9+ (or Colab)
- Packages: qdrant-client, sentence-transformers, google.colab (if using Colab)

Models
- SentenceTransformer: all-MiniLM-L6-v2 (384-dim)

Dataset

Pick something with rich, descriptive text where semantic search adds value:

- *Books/Literature*: Search a collection of book summaries, reviews, or excerpts. Find books by theme, mood, or literary style. Example queries: “coming of age stories with unreliable narrators”, “dystopian fiction with environmental themes”
- *Recipes/Cooking*: Index recipe descriptions and instructions. Search by cooking technique, flavor profile, or dietary needs. Example queries: “comfort food for cold weather”, “quick weeknight meals with Asian flavors”
- *News/Articles*: Collect articles from your field of interest. Search by topic, perspective, or journalistic approach. Example queries: “analysis of remote work trends”, “climate change solutions in urban planning”
- *Research Papers*: Academic abstracts or papers from your field. Search by methodology, findings, or theoretical approach. Example queries: “machine learning applications in healthcare”, “qualitative studies on user behavior”
- *Product Reviews*: Customer reviews for products you know well. Search by user sentiment, use case, or product features. Example queries: “laptops good for video editing under budget”, “skincare for sensitive skin winter routine”

# Step 1: Initialize Client

In [1]:
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
import os

client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"))

# For Colab:
# from google.colab import userdata
# client = QdrantClient(url=userdata.get("QDRANT_URL"), api_key=userdata.get("QDRANT_API_KEY"))

encoder = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\Praneeth\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3142.05it/s]


# Step 2: Prepare Your Dataset

In [6]:
# Example: Recipe collection
my_dataset = [
    {
        "title": "Classic Beef Bourguignon",
        "description": """A rich, wine-braised beef stew from Burgundy, France.
        Tender chunks of beef are slowly simmered with pearl onions, mushrooms,
        and bacon in a deep red wine sauce. The long, slow cooking process
        develops complex flavors and creates a luxurious, velvety texture.
        Perfect for cold winter evenings when you want something hearty and
        comforting. Traditionally served with crusty bread or creamy mashed
        potatoes to soak up the incredible sauce.""",
        "cuisine": "French",
        "difficulty": "Intermediate",
        "time": "3 hours"
    },
    {
        "title": "Thai Green Curry with Chicken",
        "description": """An aromatic and vibrant curry from Thailand featuring tender
        chicken in a coconut milk base infused with fragrant green curry paste, Thai basil,
        and kaffir lime leaves. The sauce balances spicy, sweet, and savory notes with
        bamboo shoots, eggplant, and bell peppers adding texture. Quick to prepare yet
        bursting with complex flavors, this dish brings the authentic taste of Bangkok
        street food to your kitchen. Serve over jasmine rice to soak up every drop of
        the creamy, spice-laden sauce.""",
        "cuisine": "Thai",
        "difficulty": "Easy",
        "time": "30 minutes"
    },
    {
        "title": "Homemade Margherita Pizza",
        "description": """The quintessential Neapolitan pizza that celebrates simplicity
        and quality ingredients. A thin, chewy crust with charred bubbles holds a bright
        San Marzano tomato sauce, creamy fresh mozzarella, and fragrant basil leaves.
        The key is high heat and minimal toppings, allowing each element to shine.
        Making the dough from scratch requires patience as it slowly ferments, developing
        complex flavors and that signature airy texture. Perfect for weekend cooking
        when you want to impress with authentic Italian technique.""",
        "cuisine": "Italian",
        "difficulty": "Intermediate",
        "time": "2 hours (plus dough rise)"
    },
    {
        "title": "Miso-Glazed Salmon with Sesame Vegetables",
        "description": """A modern Japanese-inspired dish that's both elegant and nutritious.
        Fresh salmon fillets are marinated in a sweet-savory miso glaze with mirin and sake,
        then broiled until caramelized and slightly charred at the edges. The umami-rich
        glaze creates a beautiful lacquered finish. Paired with crisp stir-fried vegetables
        tossed in sesame oil and garnished with toasted sesame seeds. This restaurant-quality
        meal comes together in under 25 minutes, making it perfect for busy weeknights when
        you don't want to sacrifice flavor or presentation.""",
        "cuisine": "Japanese",
        "difficulty": "Easy",
        "time": "25 minutes"
    },
    {
        "title": "Moroccan Lamb Tagine with Apricots",
        "description": """A fragrant North African stew that marries tender lamb with sweet
        dried apricots, aromatic spices, and preserved lemons. Slow-cooked in a traditional
        cone-shaped tagine or heavy pot, the meat becomes fall-apart tender while absorbing
        the warm spices of cinnamon, cumin, and ginger. Chickpeas add heartiness, while
        honey and apricots provide a delicate sweetness that balances the savory depth.
        The long, gentle cooking process allows the complex spice blend to fully develop.
        Serve over fluffy couscous with fresh cilantro and toasted almonds for a truly
        exotic dining experience.""",
        "cuisine": "Moroccan",
        "difficulty": "Intermediate",
        "time": "2.5 hours"
    },
    {
        "title": "Classic Chicken Caesar Salad",
        "description": """A timeless American restaurant staple that's surprisingly easy to
        master at home. Crisp romaine lettuce is tossed with a bold, creamy dressing made
        from anchovies, garlic, lemon, egg yolk, and Parmesan cheese. Topped with juicy
        grilled chicken breast, crunchy house-made croutons, and extra shaved Parmesan.
        The key is the dressing—punchy, garlicky, and perfectly emulsified. While often
        considered simple, a well-executed Caesar showcases the power of balancing strong,
        complementary flavors. Great for a light lunch or dinner that feels both indulgent
        and refreshing.""",
        "cuisine": "American",
        "difficulty": "Easy",
        "time": "20 minutes"
    },
    {
        "title": "Indian Butter Chicken (Murgh Makhani)",
        "description": """A beloved North Indian classic featuring tender chicken in a
        luxuriously creamy tomato-based sauce. The chicken is first marinated in yogurt
        and spices, then grilled or pan-fried for a smoky char before being simmered in
        a velvety sauce enriched with butter, cream, and aromatic spices like garam masala,
        fenugreek, and cardamom. The result is a perfect balance of tangy, sweet, and
        mildly spiced flavors with a silky texture. This restaurant favorite is easier to
        make at home than you'd think, and the aroma while cooking will transport you to
        the bustling streets of Delhi. Serve with naan bread and basmati rice.""",
        "cuisine": "Indian",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Spanish Paella Valenciana",
        "description": """The iconic rice dish from Valencia that's meant for sharing and
        celebrating. Saffron-infused short-grain rice is cooked with chicken, rabbit, and
        green beans in a wide, shallow pan until it develops the prized 'socarrat'—a
        crispy, caramelized bottom layer. Fresh rosemary and smoked paprika add depth,
        while the saffron lends its distinctive golden color and earthy aroma. Traditionally
        cooked over an open fire, this one-pan feast brings people together. Making paella
        is as much about the ritual and patience as it is about the ingredients. The key
        is resisting the urge to stir, allowing those delicious crispy bits to form.""",
        "cuisine": "Spanish",
        "difficulty": "Advanced",
        "time": "1.5 hours"
    },
    {
        "title": "Vietnamese Pho Bo (Beef Noodle Soup)",
        "description": """Vietnam's national dish—a deeply aromatic beef broth that takes
        hours to perfect. Beef bones are simmered with charred onions, ginger, star anise,
        cinnamon, and coriander seeds until the broth becomes rich, clear, and intensely
        flavorful. Served over silky rice noodles with thinly sliced rare beef that cooks
        in the steaming broth, then finished with fresh herbs, lime, jalapeños, and bean
        sprouts. Each bowl is customizable at the table, making it interactive and personal.
        While time-intensive, the reward is a soul-warming bowl of pure comfort that rivals
        any pho shop in Hanoi.""",
        "cuisine": "Vietnamese",
        "difficulty": "Advanced",
        "time": "4 hours"
    },
    {
        "title": "Greek Moussaka",
        "description": """A hearty, layered casserole that's Greece's answer to lasagna.
        Tender slices of eggplant and potato are layered with a rich, spiced ground lamb
        and tomato sauce flavored with cinnamon, oregano, and a hint of red wine. The
        crown jewel is the creamy béchamel sauce on top, which bakes to a golden brown.
        Each forkful delivers multiple textures and layers of Mediterranean flavor. While
        it requires some prep work and patience, moussaka is perfect for feeding a crowd
        or meal prepping for the week. The flavors actually improve after a day, making
        leftovers even more delicious.""",
        "cuisine": "Greek",
        "difficulty": "Intermediate",
        "time": "2 hours"
    },
    {
        "title": "Korean Bibimbap with Gochujang",
        "description": """A colorful Korean rice bowl that's as beautiful as it is delicious.
        Warm rice is topped with an array of seasoned vegetables—sautéed spinach, carrots,
        bean sprouts, mushrooms—along with marinated beef, a fried egg, and a generous
        dollop of spicy-sweet gochujang sauce. The name means 'mixed rice,' and the ritual
        of stirring everything together before eating is essential. Each ingredient is
        prepared separately, showcasing different cooking techniques and seasonings. The
        result is a harmonious bowl where every bite offers different flavors and textures.
        Customizable and nutritious, it's perfect for using up vegetables and experimenting
        with Korean flavors.""",
        "cuisine": "Korean",
        "difficulty": "Intermediate",
        "time": "1 hour"
    },
    {
        "title": "Mexican Carnitas Tacos",
        "description": """Authentic slow-cooked pork that's tender on the inside with
        crispy, caramelized edges. A pork shoulder is braised low and slow in its own
        fat with orange juice, garlic, and Mexican spices until it's so tender it falls
        apart with a fork. Then it's crisped under the broiler for textural contrast.
        Tucked into warm corn tortillas and topped with fresh cilantro, diced onions,
        lime, and your favorite salsa. These tacos are a weekend project worth every
        minute—the kind of food that brings everyone to the table. Serve with refried
        beans and Mexican rice for an authentic taqueria experience at home.""",
        "cuisine": "Mexican",
        "difficulty": "Easy",
        "time": "3.5 hours (mostly hands-off)"
    }
]

# Step 3: Implement Three Chunking Strategies

In [7]:
def fixed_size_chunks(text, chunk_size=100, overlap=20):
    """Split text into fixed-size chunks with overlap"""
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size - overlap):
        chunk_words = words[i:i + chunk_size]
        if chunk_words:  # Only add non-empty chunks
            chunks.append(' '.join(chunk_words))
    
    return chunks

def sentence_chunks(text, max_sentences=3):
    """Group sentences into chunks"""
    import re
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk_sentences = sentences[i:i + max_sentences]
        if chunk_sentences:
            chunks.append('. '.join(chunk_sentences) + '.')
    
    return chunks

def paragraph_chunks(text):
    """Split by paragraphs or double line breaks"""
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]
    return chunks if chunks else [text]  # Fallback to full text

# Step 4: Create Collections and Process Data

In [8]:
collection_name = "day1_semantic_search"

if client.collection_exists(collection_name=collection_name):
    client.delete_collection(collection_name=collection_name)

# Create a collection with three named vectors
client.create_collection(
    collection_name=collection_name,
    vectors_config={
        "fixed": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "sentence": models.VectorParams(size=384, distance=models.Distance.COSINE),
        "paragraph": models.VectorParams(size=384, distance=models.Distance.COSINE),
    },
)

# Index fields for filtering (more on this on day 2)
client.create_payload_index(
    collection_name=collection_name,
    field_name="chunk_strategy",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

# Process and upload data
points = []
point_id = 0

for item in my_dataset:
    description = item["description"]

    # Process with each chunking strategy
    strategies = {
        "fixed": fixed_size_chunks(description),
        "sentence": sentence_chunks(description),
        "paragraph": paragraph_chunks(description),
    }

    for strategy_name, chunks in strategies.items():
        for chunk_idx, chunk in enumerate(chunks):
            # Create vectors for this chunk
            vectors = {strategy_name: encoder.encode(chunk).tolist()}

            points.append(
                models.PointStruct(
                    id=point_id,
                    vector=vectors,
                    payload={
                        **item,  # Include all original metadata
                        "chunk": chunk,
                        "chunk_strategy": strategy_name,
                        "chunk_index": chunk_idx,
                    },
                )
            )
            point_id += 1

client.upload_points(collection_name=collection_name, points=points)
print(f"Uploaded {len(points)} chunks across three strategies")

Uploaded 56 chunks across three strategies


# Step 5: Test and Compare

In [9]:
def compare_search_results(query):
    """Compare search results across all chunking strategies"""
    print(f"Query: '{query}'\n")

    for strategy in ["fixed", "sentence", "paragraph"]:
        results = client.query_points(
            collection_name=collection_name,
            query=encoder.encode(query).tolist(),
            using=strategy,
            limit=3,
        )

        print(f"--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            print(f"{i}. {point.payload['title']}")
            print(f"   Score: {point.score:.3f}")
            print(f"   Chunk: {point.payload['chunk'][:80]}...")
        print()


# Test with domain-specific queries
test_queries = [
    "comfort food for winter",  # Adapt these to your domain
    "quick and easy weeknight dinner",
    "elegant dish for special occasions",
]

for query in test_queries:
    compare_search_results(query)

Query: 'comfort food for winter'

--- FIXED CHUNKING ---
1. Moroccan Lamb Tagine with Apricots
   Score: 0.411
   Chunk: almonds for a truly exotic dining experience....
2. Moroccan Lamb Tagine with Apricots
   Score: 0.391
   Chunk: A fragrant North African stew that marries tender lamb with sweet dried apricots...
3. Indian Butter Chicken (Murgh Makhani)
   Score: 0.366
   Chunk: A beloved North Indian classic featuring tender chicken in a luxuriously creamy ...

--- SENTENCE CHUNKING ---
1. Classic Beef Bourguignon
   Score: 0.557
   Chunk: Perfect for cold winter evenings when you want something hearty and
        comf...
2. Moroccan Lamb Tagine with Apricots
   Score: 0.370
   Chunk: A fragrant North African stew that marries tender lamb with sweet
        dried ...
3. Spanish Paella Valenciana
   Score: 0.355
   Chunk: The iconic rice dish from Valencia that's meant for sharing and
        celebrat...

--- PARAGRAPH CHUNKING ---
1. Moroccan Lamb Tagine with Apricots
   Score: 0.3

# Step 6: Analyze Your Results

In [10]:
def analyze_chunking_effectiveness():
    """Analyze which chunking strategy works best for your domain"""

    print("CHUNKING STRATEGY ANALYSIS")
    print("=" * 40)

    # Get chunk statistics for each strategy
    for strategy in ["fixed", "sentence", "paragraph"]:
        # Count chunks per strategy
        results = client.scroll(
            collection_name=collection_name,
            scroll_filter=models.Filter(
                must=[
                    models.FieldCondition(
                        key="chunk_strategy", match=models.MatchValue(value=strategy)
                    )
                ]
            ),
            limit=100,
        )

        chunks = results[0]
        chunk_sizes = [len(chunk.payload["chunk"]) for chunk in chunks]

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Total chunks: {len(chunks)}")
        print(f"  Avg chunk size: {sum(chunk_sizes)/len(chunk_sizes):.0f} chars")
        print(f"  Size range: {min(chunk_sizes)}-{max(chunk_sizes)} chars")


analyze_chunking_effectiveness()

CHUNKING STRATEGY ANALYSIS

FIXED STRATEGY:
  Total chunks: 20
  Avg chunk size: 373 chars
  Size range: 15-661 chars

SENTENCE STRATEGY:
  Total chunks: 24
  Avg chunk size: 305 chars
  Size range: 87-466 chars

PARAGRAPH STRATEGY:
  Total chunks: 12
  Avg chunk size: 620 chars
  Size range: 485-725 chars
